# 低評価とその後の購買行動の関係分析

## 目的

低評価となった注文が、その後の顧客の再購入や売上と
どのような関係を持つかを確認する。

低評価予測モデルを作成するビジネス上の意味を検討するため、
レビュー評価と将来の購買行動との関連を分析する。

## 分析指標

各注文を基準として、その後90日間について以下を算出する。

- `future_90d_repeat`
  - 90日以内に再購入したか
- `future_90d_order_count`
  - 90日以内の注文回数
- `future_90d_gmv`
  - 90日以内の購入金額合計

低評価はこれまでと同様、

- `low_review = 1`：review_score <= 2
- `low_review = 0`：review_score >= 3

と定義する。

## 注意点

データ収集期間の終了直前の注文では、
その後90日間を完全に観察できない。

そのため、
データの最終購入日から90日以上前の注文のみを
分析対象とする。

In [1]:
import sys
from pathlib import Path

import pandas as pd

sys.path.insert(0, str(Path.cwd().parent))

import src.project_paths

df_review_anchor = pd.read_csv(
    src.project_paths.DATA_DIR / "review_anchor.csv"
)

print("Shape:", df_review_anchor.shape)

print("order_id重複:",df_review_anchor["order_id"].duplicated().sum())

print("\n欠損数")
print(df_review_anchor.isna().sum())

print("\n評価分布")
print(df_review_anchor["review_score"].value_counts().sort_index())

print("\n低評価率")
print(df_review_anchor["low_review"].value_counts(normalize=True))

Shape: (94865, 4)
order_id重複: 0

欠損数
order_id            0
review_score        0
review_timestamp    0
low_review          0
dtype: int64

評価分布
review_score
1    10340
2     2910
3     7770
4    18560
5    55285
Name: count, dtype: int64

低評価率
low_review
0    0.860328
1    0.139672
Name: proportion, dtype: float64


In [2]:
df_future = pd.read_csv(
    src.project_paths.DATA_DIR / "future_90d_purchase_features.csv"
)

print("Shape:", df_future.shape)

print("order_id重複:", df_future["order_id"].duplicated().sum())

print("\n欠損数")
print(df_future.isna().sum())

print("\n90日以内の再購入")
print(df_future["future_90d_repeat"].value_counts())

print("\n低評価別の再購入率・注文数・GMV")
summary = df_future.groupby("low_review").agg(
    order_count=("order_id", "count"),
    repeat_rate=("future_90d_repeat", "mean"),
    avg_future_orders=("future_90d_order_count", "mean"),
    avg_future_gmv=("future_90d_gmv", "mean"),
)

print(summary)


Shape: (74185, 7)
order_id重複: 0

欠損数
order_id                  0
review_score              0
low_review                0
review_timestamp          0
future_90d_repeat         0
future_90d_order_count    0
future_90d_gmv            0
dtype: int64

90日以内の再購入
future_90d_repeat
0    73291
1      894
Name: count, dtype: int64

低評価別の再購入率・注文数・GMV
            order_count  repeat_rate  avg_future_orders  avg_future_gmv
low_review                                                             
0                 63206     0.012499           0.014413        1.662682
1                 10979     0.009473           0.010019        1.504917


これを見ると、低評価した人は、再購入率が低く、将来のGMVも低い傾向があるように見えるので、統計的にその仮説が正しいかどうかを、Fisherの正確確立検定を用いて検証する。

In [3]:
import numpy as np
import pandas as pd
from scipy.stats import fisher_exact

# 評価群 × 再購入有無のクロス集計表
contingency_table = pd.crosstab(df_future["low_review"], df_future["future_90d_repeat"])

print("クロス集計表")
print(contingency_table)

# Fisherの正確確率検定
odds_ratio, p_value = fisher_exact(contingency_table)

print("\nFisher's Exact Test")
print(f"Odds Ratio: {odds_ratio:.4f}")
print(f"p-value: {p_value:.6g}")


# 95%信頼区間の計算
group_summary = df_future.groupby("low_review")["future_90d_repeat"].agg(
    ["sum", "count", "mean"]
)

normal = group_summary.loc[0]
low = group_summary.loc[1]

# 低評価群 − 通常評価群
risk_difference = low["mean"] - normal["mean"]

# まずは単純な二項分布近似による信頼区間
se = np.sqrt(
    low["mean"] * (1 - low["mean"]) / low["count"]
    + normal["mean"] * (1 - normal["mean"]) / normal["count"]
)

ci_lower = risk_difference - 1.96 * se
ci_upper = risk_difference + 1.96 * se

print(f"再購入率の差: {risk_difference * 100:.4f} ポイント")
print(f"95%信頼区間: [{ci_lower * 100:.4f}, {ci_upper * 100:.4f}] ポイント")


クロス集計表
future_90d_repeat      0    1
low_review                   
0                  62416  790
1                  10875  104

Fisher's Exact Test
Odds Ratio: 0.7556
p-value: 0.00685284
再購入率の差: -0.3026 ポイント
95%信頼区間: [-0.5034, -0.1018] ポイント


この結果から、低評価群の再購入オッズが約24%低いこと、低評価の注文はその後90日以内の再購入が有意に低いことが確認された。再購入率と低評価の関係を検証するために、ロジスティック回帰を用いて検証する。

In [4]:
# 今回の関連分析で使う候補特徴量
adjustment_features = [
    "order_id",
    "total_price",
    "total_freight",
    "delivery_delay_days",
    "purchase_month",
    "customer_state",
    "prior_order_count",
]

# Notebook 11で保存した特徴量データを読み込む
df_model_improved_6 = pd.read_csv(
    src.project_paths.DATA_DIR / "model_improved_6.csv"
)

# 必要な列だけを取得
df_adjustment = df_model_improved_6[adjustment_features].copy()

# order_id の重複確認
print("調整用データのorder_id重複数:", df_adjustment["order_id"].duplicated().sum())

# 再購入データへ結合
df_repeat_analysis = df_future.merge(
    df_adjustment,
    on="order_id",
    how="left",
    validate="one_to_one",
    indicator=True,
)

print("\n結合後のShape:")
print(df_repeat_analysis.shape)

print("\n結合状況:")
print(df_repeat_analysis["_merge"].value_counts())

print("\n各列の欠損数:")
print(df_repeat_analysis[adjustment_features[1:]].isna().sum())


調整用データのorder_id重複数: 0

結合後のShape:
(74185, 14)

結合状況:
_merge
both          71949
left_only      2236
right_only        0
Name: count, dtype: int64

各列の欠損数:
total_price            2236
total_freight          2236
delivery_delay_days    2236
purchase_month         2236
customer_state         2236
prior_order_count      2236
dtype: int64


In [5]:
# 欠損データの分析
missing_summary = df_repeat_analysis.groupby("_merge", observed=True).agg(
    order_count=("order_id", "count"),
    low_review_rate=("low_review", "mean"),
    repeat_rate=("future_90d_repeat", "mean"),
    avg_future_gmv=("future_90d_gmv", "mean"),
)

print(missing_summary)

# 低評価率と欠損データの関係
print(
    pd.crosstab(
        df_repeat_analysis["low_review"],
        df_repeat_analysis["_merge"],
        margins=True,
    )
)

# 欠損データを除外して解析する
df_complete = (
    df_repeat_analysis[df_repeat_analysis["_merge"] == "both"]
    .drop(columns="_merge")
    .copy()
)

print("調整解析候補の件数:", len(df_complete))

print(
    df_complete.groupby("low_review").agg(
        order_count=("order_id", "count"),
        repeat_count=("future_90d_repeat", "sum"),
        repeat_rate=("future_90d_repeat", "mean"),
    )
)



           order_count  low_review_rate  repeat_rate  avg_future_gmv
_merge                                                              
left_only         2236         0.773256     0.012075        2.648739
both             71949         0.128563     0.012050        1.607964
_merge      left_only   both    All
low_review                         
0                 507  62699  63206
1                1729   9250  10979
All              2236  71949  74185
調整解析候補の件数: 71949
            order_count  repeat_count  repeat_rate
low_review                                        
0                 62699           781     0.012456
1                  9250            86     0.009297


In [6]:
from src.project_paths import DATA_DIR

df_customer_history = pd.read_csv(DATA_DIR / "customer_history_features.csv")

df_check = df_future[["order_id", "low_review", "future_90d_repeat"]].merge(
    df_customer_history[["order_id", "customer_unique_id", "prior_order_count"]],
    on="order_id",
    how="left",
    validate="one_to_one",
    indicator=True,
)

print("結合後:", df_check.shape)

print("\n結合状況")
print(df_check["_merge"].value_counts())

print("\n低評価別の結合状況")
print(
    pd.crosstab(
        df_check["low_review"],
        df_check["_merge"],
    )
)


結合後: (74185, 6)

結合状況
_merge
both          73580
left_only       605
right_only        0
Name: count, dtype: int64

低評価別の結合状況
_merge      left_only   both
low_review                  
0                 125  63081
1                 480  10499


この結果を確認すると、sqlの時点でorder_itemが存在しないデータを削除していることが、不一致の原因だと考えられる。また、不一致しているもののうちの低評価率が高いため、order_itemが存在しないデータも抽出して検討する必要性がある。

In [7]:
# 再購入解析の調整用データを読み込む
df_adjustments = pd.read_csv(DATA_DIR / "repeat_analysis_adjustments.csv")

print("Shape:", df_adjustments.shape)

print("order_id重複:", df_adjustments["order_id"].duplicated().sum())

print("\n欠損数")
print(df_adjustments.isna().sum())

print("\n注文ステータス")
print(df_adjustments["order_status"].value_counts())

print("\n商品明細の有無")
print(df_adjustments["has_order_items"].value_counts())

# 再購入データへ結合
df_repeat_adjusted = df_future.merge(
    df_adjustments,
    on="order_id",
    how="left",
    validate="one_to_one",
    indicator=True,
)

print("\n結合状況")
print(df_repeat_adjusted["_merge"].value_counts())

print("\n低評価別の結合状況")
print(
    pd.crosstab(
        df_repeat_adjusted["low_review"],
        df_repeat_adjusted["_merge"],
    )
)


Shape: (99441, 7)
order_id重複: 0

欠損数
order_id                0
customer_unique_id      0
order_status            0
purchase_timestamp      0
order_gmv             775
has_order_items         0
prior_order_count       0
dtype: int64

注文ステータス
order_status
delivered      96478
shipped         1107
canceled         625
unavailable      609
invoiced         314
processing       301
created            5
approved           2
Name: count, dtype: int64

商品明細の有無
has_order_items
1    98666
0      775
Name: count, dtype: int64

結合状況
_merge
both          74185
left_only         0
right_only        0
Name: count, dtype: int64

低評価別の結合状況
_merge       both
low_review       
0           63206
1           10979


In [8]:
import numpy as np
import pandas as pd

df_analysis = df_repeat_adjusted.copy()

df_analysis["purchase_timestamp"] = pd.to_datetime(df_analysis["purchase_timestamp"])

df_analysis["review_timestamp"] = pd.to_datetime(df_analysis["review_timestamp"])

df_analysis["purchase_month"] = (
    df_analysis["purchase_timestamp"].dt.to_period("M").astype(str)
)

print(df_analysis.shape)

print(
    df_analysis[
        [
            "low_review",
            "future_90d_repeat",
            "order_status",
            "prior_order_count",
            "purchase_month",
        ]
    ]
    .isna()
    .sum()
)


(74185, 15)
low_review           0
future_90d_repeat    0
order_status         0
prior_order_count    0
purchase_month       0
dtype: int64


In [9]:
import statsmodels.formula.api as smf

# 単純ロジスティック回帰モデル
model_unadjusted = smf.logit(
    formula="future_90d_repeat ~ low_review", data=df_analysis
).fit()

print(model_unadjusted.summary())


Optimization terminated successfully.
         Current function value: 0.065175
         Iterations 9
                           Logit Regression Results                           
Dep. Variable:      future_90d_repeat   No. Observations:                74185
Model:                          Logit   Df Residuals:                    74183
Method:                           MLE   Df Model:                            1
Date:                Mon, 19 Oct 2026   Pseudo R-squ.:               0.0007923
Time:                        01:40:58   Log-Likelihood:                -4835.0
converged:                       True   LL-Null:                       -4838.8
Covariance Type:            nonrobust   LLR p-value:                  0.005623
                 coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------
Intercept     -4.3695      0.036   -122.045      0.000      -4.440      -4.299
low_review    -0.2803      0.

In [10]:
model_adjusted_1 = smf.logit(
    formula=("future_90d_repeat ~ low_review + prior_order_count + C(purchase_month)"),
    data=df_analysis,
).fit(maxiter=200)

print(model_adjusted_1.summary())


         Current function value: 0.063157
         Iterations: 200


/usr/local/lib/python3.12/site-packages/statsmodels/discrete/discrete_model.py:268: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  mlefit = super().fit(


                           Logit Regression Results                           
Dep. Variable:      future_90d_repeat   No. Observations:                74185
Model:                          Logit   Df Residuals:                    74161
Method:                           MLE   Df Model:                           23
Date:                Mon, 19 Oct 2026   Pseudo R-squ.:                 0.03173
Time:                        01:41:02   Log-Likelihood:                -4685.3
converged:                      False   LL-Null:                       -4838.8
Covariance Type:            nonrobust   LLR p-value:                 1.723e-51
                                   coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------------------------
Intercept                      -15.2639   1161.158     -0.013      0.990   -2291.091    2260.563
C(purchase_month)[T.2016-10]    -8.5437   8955.246     -0.001      0.999   -1

In [11]:
model_adjusted_2 = smf.logit(
    formula=(
        "future_90d_repeat ~ "
        "low_review + "
        "prior_order_count + "
        "C(purchase_month) + "
        "C(order_status)"
    ),
    data=df_analysis,
).fit(maxiter=200)

print(model_adjusted_2.summary())


         Current function value: 0.063127
         Iterations: 200


/usr/local/lib/python3.12/site-packages/statsmodels/discrete/discrete_model.py:268: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  mlefit = super().fit(


                           Logit Regression Results                           
Dep. Variable:      future_90d_repeat   No. Observations:                74185
Model:                          Logit   Df Residuals:                    74154
Method:                           MLE   Df Model:                           30
Date:                Mon, 19 Oct 2026   Pseudo R-squ.:                 0.03218
Time:                        01:41:10   Log-Likelihood:                -4683.1
converged:                      False   LL-Null:                       -4838.8
Covariance Type:            nonrobust   LLR p-value:                 1.439e-48
                                     coef    std err          z      P>|z|      [0.025      0.975]
--------------------------------------------------------------------------------------------------
Intercept                        -26.3828   2143.635     -0.012      0.990   -4227.830    4175.064
C(purchase_month)[T.2016-10]      -8.2236   8955.442     -0.001      0.

In [12]:
import src.model_summary

or_results = pd.DataFrame(
    [
        src.model_summary.extract_low_review_or(model_unadjusted, "Unadjusted"),
        src.model_summary.extract_low_review_or(model_adjusted_1, "Adjusted 1"),
        src.model_summary.extract_low_review_or(model_adjusted_2, "Adjusted 2"),
    ]
)

print(or_results)


        model  odds_ratio  ci_lower  ci_upper   p_value
0  Unadjusted    0.755567  0.615236  0.927906  0.007501
1  Adjusted 1    0.789199  0.640916  0.971790  0.025784
2  Adjusted 2    0.765220  0.614292  0.953230  0.016973


In [13]:
month_summary = df_analysis.groupby("purchase_month").agg(
    order_count=("order_id", "count"),
    repeat_count=("future_90d_repeat", "sum"),
    repeat_rate=("future_90d_repeat", "mean"),
)

print("=== 購入月別 ===")
print(month_summary)

status_summary = df_analysis.groupby("order_status").agg(
    order_count=("order_id", "count"),
    repeat_count=("future_90d_repeat", "sum"),
    repeat_rate=("future_90d_repeat", "mean"),
    low_review_rate=("low_review", "mean"),
)

print("\n=== 注文ステータス別 ===")
print(status_summary)


=== 購入月別 ===
                order_count  repeat_count  repeat_rate
purchase_month                                        
2016-09                   4             0     0.000000
2016-10                 293             0     0.000000
2016-12                   1             1     1.000000
2017-01                 757             4     0.005284
2017-02                1680            14     0.008333
2017-03                2550            34     0.013333
2017-04                2304            19     0.008247
2017-05                3547            50     0.014096
2017-06                3083            39     0.012650
2017-07                3845            53     0.013784
2017-08                4137            61     0.014745
2017-09                4082            62     0.015189
2017-10                4412            49     0.011106
2017-11                7181            81     0.011280
2017-12                5391            59     0.010944
2018-01                6889            83     0.0120

再購入者が０の月や、０のorder_statusがあるため、ロジスティック回帰が収束しなかった可能性が高いため、データ数を残しつつ、これらのデータを調整。

In [14]:
# 購入日を経過日数として数値化
reference_date = df_analysis["purchase_timestamp"].min()

df_analysis["purchase_days"] = (
    df_analysis["purchase_timestamp"] - reference_date
).dt.total_seconds() / 86400

# 解釈しやすいよう100日単位に変換
df_analysis["purchase_days_100"] = df_analysis["purchase_days"] / 100

# 配送完了かどうか
df_analysis["is_delivered"] = (df_analysis["order_status"] == "delivered").astype(int)

print(
    df_analysis[
        [
            "purchase_days_100",
            "is_delivered",
            "prior_order_count",
        ]
    ].describe()
)

print("\n配送完了の内訳")
print(df_analysis["is_delivered"].value_counts())


       purchase_days_100  is_delivered  prior_order_count
count       74185.000000  74185.000000       74185.000000
mean            4.299256      0.969886           0.038094
std             1.306649      0.170902           0.237951
min             0.000000      0.000000           0.000000
25%             3.336155      1.000000           0.000000
50%             4.496952      1.000000           0.000000
75%             5.389242      1.000000           0.000000
max             7.049024      1.000000          11.000000

配送完了の内訳
is_delivered
1    71951
0     2234
Name: count, dtype: int64


In [15]:
import statsmodels.formula.api as smf

# 購入時期・配送完了の有無・過去注文数を調整したロジスティック回帰モデル
model_adjusted_1_v2 = smf.logit(
    formula=("future_90d_repeat ~ low_review + purchase_days_100 + prior_order_count"),
    data=df_analysis,
).fit(maxiter=200)

print(model_adjusted_1_v2.summary())
print("収束:", model_adjusted_1_v2.mle_retvals["converged"])


Optimization terminated successfully.
         Current function value: 0.063409
         Iterations 9
                           Logit Regression Results                           
Dep. Variable:      future_90d_repeat   No. Observations:                74185
Model:                          Logit   Df Residuals:                    74181
Method:                           MLE   Df Model:                            3
Date:                Mon, 19 Oct 2026   Pseudo R-squ.:                 0.02787
Time:                        01:41:11   Log-Likelihood:                -4704.0
converged:                       True   LL-Null:                       -4838.8
Covariance Type:            nonrobust   LLR p-value:                 3.535e-58
                        coef    std err          z      P>|z|      [0.025      0.975]
-------------------------------------------------------------------------------------
Intercept            -4.4453      0.118    -37.816      0.000      -4.676      -4.215
low_revi

In [16]:
# 配送完了の有無を追加したロジスティック回帰モデル
model_adjusted_2_v2 = smf.logit(
    formula=(
        "future_90d_repeat ~ "
        "low_review + "
        "purchase_days_100 + "
        "prior_order_count + "
        "is_delivered"
    ),
    data=df_analysis,
).fit(maxiter=200)

print(model_adjusted_2_v2.summary())
print("収束:", model_adjusted_2_v2.mle_retvals["converged"])


Optimization terminated successfully.
         Current function value: 0.063401
         Iterations 9
                           Logit Regression Results                           
Dep. Variable:      future_90d_repeat   No. Observations:                74185
Model:                          Logit   Df Residuals:                    74180
Method:                           MLE   Df Model:                            4
Date:                Mon, 19 Oct 2026   Pseudo R-squ.:                 0.02798
Time:                        01:41:12   Log-Likelihood:                -4703.4
converged:                       True   LL-Null:                       -4838.8
Covariance Type:            nonrobust   LLR p-value:                 2.122e-57
                        coef    std err          z      P>|z|      [0.025      0.975]
-------------------------------------------------------------------------------------
Intercept            -4.2311      0.231    -18.334      0.000      -4.683      -3.779
low_revi

In [17]:
# 調整したロジスティック回帰モデルの結果をまとめる
or_results_v2 = pd.DataFrame(
    [
        src.model_summary.extract_low_review_or(model_unadjusted, "Unadjusted"),
        src.model_summary.extract_low_review_or(model_adjusted_1_v2, "Adjusted 1 v2"),
        src.model_summary.extract_low_review_or(model_adjusted_2_v2, "Adjusted 2 v2"),
    ]
)

print(or_results_v2)


           model  odds_ratio  ci_lower  ci_upper   p_value
0     Unadjusted    0.755567  0.615236  0.927906  0.007501
1  Adjusted 1 v2    0.769375  0.625462  0.946402  0.013089
2  Adjusted 2 v2    0.741846  0.595704  0.923839  0.007639


修正版の調整モデルは両方とも正常に収束し、購入時期・過去購入回数・配送完了状況を考慮しても、低評価群の再購入オッズが低いという関連が維持。
prior_order_countの係数が役1.034、オッズ比に変換すると約2.81。つまり、このモデルでは過去注文回数が1回多いことが、再購入オッズの約2.8倍という関連を示している。
一方、purchase_days_100 と is_delivered は、このモデルでは統計的に有意ではなかった。

ここまでの95%信頼区間とp値は、注文ごとの観測が独立であることを前提とした通常の標準誤差に基づいている。
しかし、同じ顧客が複数回登場すると、その顧客の購買傾向によって観測間に相関が生じる可能性がある。
そこで、customer_unique_id 単位でクラスターロバスト標準誤差を適用。

In [18]:
print("注文数:", len(df_analysis))

print("ユニーク顧客数:", df_analysis["customer_unique_id"].nunique())

print(
    "複数回登場する顧客数:",
    (df_analysis["customer_unique_id"].value_counts().gt(1).sum()),
)


注文数: 74185
ユニーク顧客数: 71814
複数回登場する顧客数: 2131


In [19]:
# クラスタリングを考慮したロジスティック回帰モデル
model_adjusted_2_cluster = smf.logit(
    formula=(
        "future_90d_repeat ~ "
        "low_review + "
        "purchase_days_100 + "
        "prior_order_count + "
        "is_delivered"
    ),
    data=df_analysis,
).fit(
    maxiter=200,
    cov_type="cluster",
    cov_kwds={"groups": df_analysis["customer_unique_id"]},
)

print(model_adjusted_2_cluster.summary())

print("収束:", model_adjusted_2_cluster.mle_retvals["converged"])


Optimization terminated successfully.
         Current function value: 0.063401
         Iterations 9
                           Logit Regression Results                           
Dep. Variable:      future_90d_repeat   No. Observations:                74185
Model:                          Logit   Df Residuals:                    74180
Method:                           MLE   Df Model:                            4
Date:                Mon, 19 Oct 2026   Pseudo R-squ.:                 0.02798
Time:                        01:41:33   Log-Likelihood:                -4703.4
converged:                       True   LL-Null:                       -4838.8
Covariance Type:              cluster   LLR p-value:                 2.122e-57
                        coef    std err          z      P>|z|      [0.025      0.975]
-------------------------------------------------------------------------------------
Intercept            -4.2311      0.244    -17.375      0.000      -4.708      -3.754
low_revi

In [20]:
# 低評価のodds比を通常の標準誤差と顧客単位クラスターロバストで比較
or_comparison = pd.DataFrame(
    [
        src.model_summary.extract_low_review_or(model_adjusted_2_v2, "通常の標準誤差"),
        src.model_summary.extract_low_review_or(model_adjusted_2_cluster, "顧客単位クラスターロバスト"),
    ]
)

print(or_comparison)


           model  odds_ratio  ci_lower  ci_upper   p_value
0        通常の標準誤差    0.741846  0.595704  0.923839  0.007639
1  顧客単位クラスターロバスト    0.741846  0.586682  0.938046  0.012626


顧客単位のクラスタリングによる自己相関を制御するため、顧客ID（customer_unique_id）をクラスター変数としたロバスト標準誤差を用いた。

その結果、調整後モデルにおいても低評価レビューのオッズ比は 0.742（95% CI: 0.586 - 0.938, p = 0.0126）であり、統計的に有意な負の影響が確認された。

## 低評価と90日以内の将来GMVの関連分析

### 目的

低評価と90日以内の再購入率には統計的に有意な負の関連が確認された。

次に、低評価と将来GMVの関連を検証する。

将来GMVには再購入が発生しなかった注文の0が多数含まれるため、全注文を対象とした分析と、再購入が発生した注文に限定した分析を区別する。

In [21]:
# 将来GMVの記述統計量

print("=== 全注文の将来GMV ===")

print(
    df_future["future_90d_gmv"].describe(
        percentiles=[0.25, 0.5, 0.75, 0.90, 0.95, 0.99]
    )
)

print("\n=== 将来GMVが0の注文数 ===")

print((df_future["future_90d_gmv"] == 0).sum())

print("\n=== 将来GMVが0より大きい注文数 ===")

print((df_future["future_90d_gmv"] > 0).sum())

print("\n=== 将来GMVの欠損数 ===")

print(df_future["future_90d_gmv"].isna().sum())


=== 全注文の将来GMV ===
count    74185.000000
mean         1.639334
std         22.800712
min          0.000000
25%          0.000000
50%          0.000000
75%          0.000000
90%          0.000000
95%          0.000000
99%         35.900000
max       1999.990000
Name: future_90d_gmv, dtype: float64

=== 将来GMVが0の注文数 ===
73293

=== 将来GMVが0より大きい注文数 ===
892

=== 将来GMVの欠損数 ===
0


In [22]:
# 評価群別の将来GMVの記述統計量
gmv_summary = df_future.groupby("low_review")["future_90d_gmv"].describe(
    percentiles=[0.50, 0.75, 0.90, 0.95, 0.99]
)

print(gmv_summary)


              count      mean        std  min  50%  75%  90%  95%   99%  \
low_review                                                                
0           63206.0  1.662682  22.059709  0.0  0.0  0.0  0.0  0.0  39.9   
1           10979.0  1.504917  26.669929  0.0  0.0  0.0  0.0  0.0   0.0   

                max  
low_review           
0           1369.90  
1           1999.99  


In [23]:
# 再購入が発生した注文だけで比較

df_repeat_only = df_future[df_future["future_90d_repeat"] == 1].copy()

print("再購入が発生した注文数:", len(df_repeat_only))

print("\n=== 再購入があった注文の将来GMV ===")

print(
    df_repeat_only.groupby("low_review")["future_90d_gmv"].describe(
        percentiles=[0.25, 0.50, 0.75, 0.90, 0.95]
    )
)

再購入が発生した注文数: 894

=== 再購入があった注文の将来GMV ===
            count        mean         std  min   25%    50%      75%      90%  \
low_review                                                                      
0           790.0  133.027215  146.579949  0.0  47.9   93.3  154.975  281.579   
1           104.0  158.870000  224.871628  0.0  59.6  107.4  196.370  249.963   

                95%      max  
low_review                    
0           396.000  1369.90  
1           413.875  1999.99  


In [41]:
import src.bootstrap

# 全注文の平均将来GMVの差
result_all = src.bootstrap.bootstrap_mean_difference(
    data=df_future,
    group_col="low_review",
    value_col="future_90d_gmv",
)

print("=== 全注文 ===")
print(result_all)

# 再購入に限定した平均将来GMVの差
result_repeat = src.bootstrap.bootstrap_mean_difference(
    data=df_repeat_only,
    group_col="low_review",
    value_col="future_90d_gmv",
)

print("=== 再購入あり ===")
print(result_repeat)

=== 全注文 ===
{'mean_difference': np.float64(-0.1577656812077317), 'ci_lower': np.float64(-0.6291197882199316), 'ci_upper': np.float64(0.42410892978815257)}
=== 再購入あり ===
{'mean_difference': np.float64(25.842784810126574), 'ci_lower': np.float64(-12.38004958617334), 'ci_upper': np.float64(76.58115027994153)}


全注文の平均将来GMVでは、低評価群の平均将来GMVは約0.158低い一方、再購入に限定すると、低評価群の平均将来GMVは約25.843高い（再購入はサンプルサイズが小さい）。

どちらの差も統計的な有意差は見られなかった。これまでの結果をまとめると、以下のようになる。

90日以内再購入率	    低評価群で約0.30ポイント低い	統計的に有意
全注文の平均将来GMV	    低評価群で約0.16 BRL低い	    明確な差なし
再購入時の平均将来GMV	低評価群で約25.84 BRL高い	    明確な差なし

続いて、90日以内の再購入率と同様に、顧客単位のクラスターブートストラップを行う。

In [30]:
df_gmv_analysis = df_future.merge(
    df_adjustments[["order_id", "customer_unique_id"]],
    on="order_id",
    how="left",
    validate="one_to_one",
)

print("結合後:", df_gmv_analysis.shape)

print("customer_unique_idの欠損数:", df_gmv_analysis["customer_unique_id"].isna().sum())

print("ユニーク顧客数:", df_gmv_analysis["customer_unique_id"].nunique())

print(
    "複数回登場する顧客数:",
    df_gmv_analysis["customer_unique_id"].value_counts().gt(1).sum(),
)


結合後: (74185, 8)
customer_unique_idの欠損数: 0
ユニーク顧客数: 71814
複数回登場する顧客数: 2131


これを見ると、複数回会登場する顧客数（2131人）の影響を排除したクラスターブートストラップを実装する必要がある。

In [42]:
# 顧客単位クラスターブートストラップ
cluster_results = src.bootstrap.cluster_bootstrap_gmv_difference(
    data=df_gmv_analysis,
    n_boot=2000,
    random_state=42,
)

print(cluster_results)


             analysis  mean_difference   ci_lower   ci_upper
0          All orders        -0.157766  -0.655070   0.429992
1  Repeat orders only        25.842785 -12.247393  73.609259
